<a href="https://colab.research.google.com/github/davidrpugh/introduction-to-deep-learning/blob/master/notebooks/02c-building-an-image-classifier-with-pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building an Image Classifier with PyTorch

In [1]:
%%bash

pip install --upgrade torchmetrics

In [2]:
import pathlib


import torch
from torch import nn, optim, utils
import torchmetrics
import torchvision
import torchvision.transforms.v2 as T


# default linewidth is 80 characters
torch.set_printoptions(linewidth=120)


## Verifying availability of GPU(s)

In [3]:
print(torch.__version__)

2.8.0+cu126


In [4]:
%%bash

nvidia-smi

Mon Nov 10 12:16:07 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
print(torch.cuda.is_available())

True


In [6]:
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [7]:
print(DEVICE)

cuda


## Loading the Fashion MNIST dataset using TorchVision

[TorchVision](https://docs.pytorch.org/vision/stable/index.html) is a core PyTorch library for computer vision. Torchvision provides:

* Tools to download common datasets (e.g., MNIST, FashionMNIST).
* Pretrained models for vision tasks.
* Image transformations (crop, rotate, resize, etc.).

TorchVision is preinstalled on Google Colab and Kaggle making it easy to use in teaching and research.

### Fashion MNIST

The Fashion MNIST dataset has the same structure of the familiar MNIST dataset.

* 60,000 training images
* 10,000 test images
* Images are single channel (i.e., grayscale) images with 28 x 28 = 784 pixels.

### Image Preprocessing with Transforms

* TorchVision datasets accept a `transform` argument for preprocessing.
* Common transforms: scaling, normalization, cropping, etc.
* Use `Compose` to chain multiple transforms.
* `ToImage`: converts input to a Tensor image.
* `ToDtype`: converts to float32 and scales pixel values to [0.0, 1.0].

**Be sure to use version 2 of the TorchVision transforms (i.e., `torchvision.transforms.v2`)! Version 2 is much faster, has more transforms and features, and is backward-compatible with version 1.**

In [8]:
DATA_DIR = pathlib.Path("./sample_data")


to_tensor = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
])


train_val_dataset = (
    torchvision.datasets
               .FashionMNIST(
                   DATA_DIR,
                   train=True,
                   download=True,
                   transform=to_tensor
               )
)

test_dataset = (
    torchvision.datasets
               .FashionMNIST(
                   DATA_DIR,
                   train=False,
                   download=True,
                   transform=to_tensor
               )
)

In [9]:
%%bash

ls ./sample_data/FashionMNIST/raw

t10k-images-idx3-ubyte
t10k-images-idx3-ubyte.gz
t10k-labels-idx1-ubyte
t10k-labels-idx1-ubyte.gz
train-images-idx3-ubyte
train-images-idx3-ubyte.gz
train-labels-idx1-ubyte
train-labels-idx1-ubyte.gz


In [10]:
X0, y0 = train_val_dataset[0]
print(X0.shape)
print(X0.dtype)

torch.Size([1, 28, 28])
torch.float32


In [11]:
train_val_dataset.classes[y0]

'Ankle boot'

## Prepare the data

### Train/Val split

In [12]:
_ = torch.manual_seed(42)

train_dataset, val_dataset = (
    utils.data
         .random_split(
             train_val_dataset,
             [55_000, 5_000]
         )
)

### Create the DataLoaders

In [13]:
data_loader_kwargs = {
    "batch_size": 32,
    "num_workers": 2,            # load data in parallel using multiple workers
    "persistent_workers": True,  # keep workers around between epochs
    "pin_memory": True,          # avoid extra copy of data batches
    "prefetch_factor": 2,        # fetch multiple data batches in advance
}


train_data_loader = (
    utils.data
         .DataLoader(
             train_dataset,
             shuffle=True,
             **data_loader_kwargs
         )
)

val_data_loader = (
    utils.data
         .DataLoader(
             val_dataset,
             shuffle=False,
             **data_loader_kwargs
         )
)

test_data_loader = (
    utils.data
         .DataLoader(
             test_dataset,
             shuffle=False,
             **data_loader_kwargs
         )
)

### Wrapping our model in a custom module

In [14]:
class MLPClassifier(nn.Module):

    def __init__(self, input_size, hidden_layer_sizes, n_classes):
        super().__init__()

        # create the hidden layers
        modules = nn.ModuleList([nn.Flatten()])
        for hidden_layer_size in hidden_layer_sizes:
            modules.append(nn.Linear(input_size, hidden_layer_size))
            modules.append(nn.ReLU())
            input_size = hidden_layer_size

        # define the output layer for the classifier
        modules.append(nn.Linear(input_size, n_classes))

        # create the MLP from the modules
        self.mlp = nn.Sequential(*modules)

    def forward(self, X):
        return self.mlp(X)



## Defining the training and evaluation loop

In [15]:
def evaluate(model_fn, data_loader, metric):
    model_fn.eval()
    metric.reset()  # reset the metric at the beginning
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            y_pred = model_fn(X_batch)
            metric.update(y_pred, y_batch)  # update it at each iteration
    return metric.compute()  # compute the final result at the end


def train(
    model_fn,
    criterion,
    optimizer,
    metric,
    train_data_loader,
    val_data_loader,
    n_epochs,
    log_epochs=1,
    ):

    history = {
        "train_losses": [],
        "val_losses": [],
        "train_metrics": [],
        "val_metrics": [],
    }

    for epoch in range(n_epochs):
        total_train_loss = 0.0
        metric.reset()
        for i, (X_batch, y_batch) in enumerate(train_data_loader):
            model_fn.train()

            # move batches to device
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            # forward pass
            y_pred = model_fn(X_batch)
            train_loss = criterion(y_pred, y_batch)
            total_train_loss += train_loss.item()

            # backward pass
            train_loss.backward()

            # gradient descent step
            optimizer.step()
            optimizer.zero_grad()

            # update our metric
            metric.update(y_pred, y_batch)

        # comute the average (across batches!) training loss
        average_train_loss = total_train_loss / len(train_data_loader)
        history["train_losses"].append(average_train_loss)

        # compute the average (across batched!) validation loss
        with torch.no_grad():
            model_fn.eval()
            total_val_loss = 0.0
            for X_batch, y_batch in val_data_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                y_pred = model_fn(X_batch)
                val_loss = criterion(y_pred, y_batch)
                total_val_loss += val_loss.item()
            average_val_loss = total_val_loss / len(val_data_loader)
            history["val_losses"].append(average_val_loss)

        # compute the training metric after each epoch
        average_train_metric = (
            metric.compute()
                  .item()
        )
        history["train_metrics"].append(average_train_metric)

        # compute the validation metric after each epoch
        average_val_metric = evaluate(
            model_fn,
            val_data_loader,
            metric,
        )
        history["val_metrics"].append(average_val_metric)

        if (epoch + 1) % log_epochs == 0:
            print(f"Epoch {epoch + 1}/{n_epochs}, "
                  f"train loss: {history['train_losses'][-1]:.4f}, "
                  f"val loss: {history['val_losses'][-1]:.4f}, "
                  f"train metric: {history['train_metrics'][-1]:.4f}, "
                  f"val metric: {history['val_metrics'][-1]:.4f}"
            )

    return history


## Putting everything together!

In [16]:
_ = torch.manual_seed(42)

# define the model function
fashion_mnist_model_fn = MLPClassifier(
    input_size=28 * 28,
    hidden_layer_sizes=[256, 128],
    n_classes=10
)
fashion_mnist_model_fn = fashion_mnist_model_fn.to(DEVICE)

# select loss function
cross_entropy_loss = nn.CrossEntropyLoss()

# define the optimizer
sgd = optim.SGD(
    fashion_mnist_model_fn.parameters(),
    lr=1e-1
)

# select a metric
accuracy = (
    torchmetrics.Accuracy(
        task="multiclass",
        num_classes=10,
    ).to(DEVICE)
)

# train your model
history = train(
    model_fn=fashion_mnist_model_fn,
    criterion=cross_entropy_loss,
    optimizer=sgd,
    metric=accuracy,
    train_data_loader=train_data_loader,
    val_data_loader=val_data_loader,
    n_epochs=20,
    log_epochs=1
)

Epoch 1/20, train loss: 0.6026, val loss: 0.7308, train metric: 0.7790, val metric: 0.7354
Epoch 2/20, train loss: 0.4043, val loss: 0.4243, train metric: 0.8501, val metric: 0.8448
Epoch 3/20, train loss: 0.3592, val loss: 0.3661, train metric: 0.8671, val metric: 0.8652
Epoch 4/20, train loss: 0.3321, val loss: 0.3552, train metric: 0.8759, val metric: 0.8676
Epoch 5/20, train loss: 0.3117, val loss: 0.3538, train metric: 0.8840, val metric: 0.8660
Epoch 6/20, train loss: 0.2959, val loss: 0.3439, train metric: 0.8896, val metric: 0.8774
Epoch 7/20, train loss: 0.2857, val loss: 0.3357, train metric: 0.8930, val metric: 0.8778
Epoch 8/20, train loss: 0.2717, val loss: 0.3276, train metric: 0.8980, val metric: 0.8826
Epoch 9/20, train loss: 0.2612, val loss: 0.3269, train metric: 0.9018, val metric: 0.8824
Epoch 10/20, train loss: 0.2521, val loss: 0.3712, train metric: 0.9037, val metric: 0.8662
Epoch 11/20, train loss: 0.2425, val loss: 0.4073, train metric: 0.9073, val metric: 0.84

## Predicting using the training model

In [17]:
def predict(X, model_fn):
    model_fn.eval()
    with torch.no_grad():
        y_pred_logits = model_fn(X)
    class_indices = torch.argmax(y_pred_logits, dim=1)
    return class_indices


def predict_proba(X, model_fn):
    model_fn.eval()
    with torch.no_grad():
        y_pred_logits = model_fn(X)
        y_pred_proba = torch.softmax(y_pred_logits, dim=1)
    return y_pred_proba


In [18]:
X_new, y_new = next(iter(val_data_loader))

In [19]:
X_new.device

device(type='cpu')

In [20]:
X_new = X_new.to(DEVICE)

In [21]:
class_indices = predict(X_new, fashion_mnist_model_fn)
print(class_indices)

tensor([7, 4, 4, 5, 9, 8, 7, 7, 7, 4, 4, 4, 7, 7, 6, 0, 1, 3, 7, 1, 2, 4, 4, 0, 6, 2, 4, 8, 5, 7, 2, 9],
       device='cuda:0')


In [22]:
class_labels = [train_val_dataset.classes[i] for i in class_indices]
print(class_labels)

['Sneaker', 'Coat', 'Coat', 'Sandal', 'Ankle boot', 'Bag', 'Sneaker', 'Sneaker', 'Sneaker', 'Coat', 'Coat', 'Coat', 'Sneaker', 'Sneaker', 'Shirt', 'T-shirt/top', 'Trouser', 'Dress', 'Sneaker', 'Trouser', 'Pullover', 'Coat', 'Coat', 'T-shirt/top', 'Shirt', 'Pullover', 'Coat', 'Bag', 'Sandal', 'Sneaker', 'Pullover', 'Ankle boot']


In [23]:
class_probas = predict_proba(X_new, fashion_mnist_model_fn)
print(class_probas.round(decimals=3))

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.9630, 0.0000, 0.0370],
        [0.0000, 0.0000, 0.0310, 0.0000, 0.9690, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0160, 0.0000, 0.2220, 0.0040, 0.6900, 0.0000, 0.0490, 0.0000, 0.0190, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0010, 0.0000, 0.9990],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0020, 0.0000, 0.8270, 0.0030, 0.1680],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.9490, 0.0000, 0.0510],
        [0.0000, 0.0010, 0.0070, 0.0360, 0.9550, 0.0000, 0.0010, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0010, 0.4240, 0.5660, 0.0000, 0.0090, 0.0000, 0.0000, 0.0000],
        [0

In [24]:
def predict_topk(X, model_fn, k=3):
    model_fn.eval()
    with torch.no_grad():
        y_pred_logits = model_fn(X)
        _, topk_class_indices = torch.topk(y_pred_logits, k=k, dim=1)
    return topk_class_indices


def predict_topk_proba(X, model_fn, k=3):
    model_fn.eval()
    with torch.no_grad():
        y_pred_logits = model_fn(X)
        topk_logits, topk_class_indices = torch.topk(y_pred_logits, k=k, dim=1)
        topk_probas = torch.softmax(topk_logits, dim=1)
    return topk_probas



In [25]:
top3_class_indices = predict_topk(X_new, fashion_mnist_model_fn, k=3)
print(top3_class_indices)

tensor([[7, 9, 5],
        [4, 2, 6],
        [4, 2, 6],
        [5, 0, 9],
        [9, 7, 5],
        [8, 4, 3],
        [7, 9, 5],
        [7, 9, 8],
        [7, 9, 5],
        [4, 3, 2],
        [4, 3, 6],
        [4, 0, 2],
        [7, 5, 3],
        [7, 5, 9],
        [6, 4, 3],
        [0, 6, 2],
        [1, 0, 3],
        [3, 0, 6],
        [7, 9, 5],
        [1, 3, 0],
        [2, 4, 0],
        [4, 2, 6],
        [4, 3, 6],
        [0, 6, 2],
        [6, 4, 2],
        [2, 0, 4],
        [4, 3, 2],
        [8, 0, 9],
        [5, 7, 0],
        [7, 9, 8],
        [2, 6, 4],
        [9, 7, 5]], device='cuda:0')


In [26]:
top3_class_labels = []
for class_indices in top3_class_indices:
    top3_class_labels.append(
        [train_val_dataset.classes[i] for i in class_indices]
    )
print(top3_class_labels)


[['Sneaker', 'Ankle boot', 'Sandal'], ['Coat', 'Pullover', 'Shirt'], ['Coat', 'Pullover', 'Shirt'], ['Sandal', 'T-shirt/top', 'Ankle boot'], ['Ankle boot', 'Sneaker', 'Sandal'], ['Bag', 'Coat', 'Dress'], ['Sneaker', 'Ankle boot', 'Sandal'], ['Sneaker', 'Ankle boot', 'Bag'], ['Sneaker', 'Ankle boot', 'Sandal'], ['Coat', 'Dress', 'Pullover'], ['Coat', 'Dress', 'Shirt'], ['Coat', 'T-shirt/top', 'Pullover'], ['Sneaker', 'Sandal', 'Dress'], ['Sneaker', 'Sandal', 'Ankle boot'], ['Shirt', 'Coat', 'Dress'], ['T-shirt/top', 'Shirt', 'Pullover'], ['Trouser', 'T-shirt/top', 'Dress'], ['Dress', 'T-shirt/top', 'Shirt'], ['Sneaker', 'Ankle boot', 'Sandal'], ['Trouser', 'Dress', 'T-shirt/top'], ['Pullover', 'Coat', 'T-shirt/top'], ['Coat', 'Pullover', 'Shirt'], ['Coat', 'Dress', 'Shirt'], ['T-shirt/top', 'Shirt', 'Pullover'], ['Shirt', 'Coat', 'Pullover'], ['Pullover', 'T-shirt/top', 'Coat'], ['Coat', 'Dress', 'Pullover'], ['Bag', 'T-shirt/top', 'Ankle boot'], ['Sandal', 'Sneaker', 'T-shirt/top'], ['

In [27]:
top3_probas = predict_topk_proba(X_new, fashion_mnist_model_fn, k=3)
print(top3_probas.round(decimals=3))

tensor([[0.9630, 0.0370, 0.0000],
        [0.9690, 0.0310, 0.0000],
        [0.7190, 0.2310, 0.0510],
        [1.0000, 0.0000, 0.0000],
        [0.9990, 0.0010, 0.0000],
        [1.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000],
        [0.8290, 0.1680, 0.0030],
        [0.9490, 0.0510, 0.0000],
        [0.9570, 0.0360, 0.0070],
        [0.5660, 0.4250, 0.0090],
        [0.9900, 0.0060, 0.0040],
        [1.0000, 0.0000, 0.0000],
        [0.9930, 0.0040, 0.0020],
        [0.5840, 0.4070, 0.0090],
        [1.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000],
        [0.9960, 0.0040, 0.0000],
        [0.9340, 0.0660, 0.0000],
        [1.0000, 0.0000, 0.0000],
        [0.8390, 0.1250, 0.0360],
        [0.9990, 0.0010, 0.0000],
        [0.7550, 0.1730, 0.0720],
        [0.8590, 0.1390, 0.0020],
        [0.6290, 0.3480, 0.0220],
        [1.0000, 0.0000, 0.0000],
        [0.9990, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000],
        [0.974